In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer

In [2]:
KOPPEN_PATH = Path("../../koppen_geiger_tif/1991_2020/koppen_geiger_0p00833333.tif")

with rasterio.open(KOPPEN_PATH) as src:
    arr = src.read(1)
    
    print("Driver:", src.driver)
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Transform:", src.transform)
    print("Dtype:", src.dtypes)
    print("NoData:", src.nodata)

nodata = src.nodata

if nodata is not None:
    valid = arr[arr != nodata]
else:
    valid = arr[~np.isnan(arr)] if np.issubdtype(arr.dtype, np.floating) else arr

unique, counts = np.unique(valid, return_counts=True)

koppen_counts = pd.DataFrame({
    "koppen_code": unique.astype(int),
    "count": counts,
    "share": counts / counts.sum()
}).sort_values("koppen_code")

display(koppen_counts)
print("Number of classes:", len(koppen_counts))

Driver: GTiff
CRS: EPSG:4326
Width: 43200
Height: 21600
Resolution: (0.008333333333333333, 0.008333333333333333)
Bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)
Transform: | 0.01, 0.00,-180.00|
| 0.00,-0.01, 90.00|
| 0.00, 0.00, 1.00|
Dtype: ('uint8',)
NoData: 0.0


,koppen_code,count,share
0,1,7992483,0.025842
1,2,5769164,0.018654
2,3,21375117,0.069113
3,4,27651140,0.089405
4,5,7951149,0.025709
5,6,10776077,0.034843
6,7,11447010,0.037012
7,8,2463440,0.007965
8,9,1477877,0.004778
9,10,9067,0.000029


Number of classes: 30


In [5]:
xmin, ymin, xmax, ymax = 18.8, 42.1, 23.1, 46.3

with rasterio.open(KOPPEN_PATH) as src:
    window = from_bounds(xmin, ymin, xmax, ymax, src.transform)
    arr = src.read(1, window=window)
    nodata = src.nodata

valid = arr[arr != nodata]

unique, counts = np.unique(valid, return_counts=True)

koppen_counts = pd.DataFrame({
    "koppen_code": unique.astype(int),
    "count": counts,
    "share": counts / counts.sum()
}).sort_values("koppen_code")

display(koppen_counts)
print("Number of classes in Serbia bbox:", len(koppen_counts))

,koppen_code,count,share
0,8,1993,0.007676
1,9,29,0.000112
2,14,139397,0.536890
3,15,17113,0.065911
4,25,2542,0.009791
5,26,94119,0.362501
6,27,4413,0.016997
7,29,32,0.000123


Number of classes in Serbia bbox: 8


In [15]:
DF_PATH = Path("../../df_valid.csv")
KOPPEN_PATH = Path("../../koppen_geiger_tif/1991_2020/koppen_geiger_0p00833333.tif")

df = pd.read_csv(DF_PATH, index_col=0)
koppen_df = df[["row", "col", "northing", "easting"]]
with rasterio.open(KOPPEN_PATH) as src:
    transformer = Transformer.from_crs("EPSG:32634", src.crs, always_xy=True)

    lon, lat = transformer.transform(
        df["easting"].to_numpy(),
        df["northing"].to_numpy(),
    )

    coords = list(zip(lon, lat))
    koppen = np.array([v[0] for v in src.sample(coords)])

    nodata = src.nodata

if nodata is not None:
    koppen = np.where(koppen == nodata, np.nan, koppen)

koppen_df["koppen_class"] = koppen

print(koppen_df["koppen_class"].value_counts(dropna=False).sort_index())
print("Missing Köppen:", koppen_df["koppen_class"].isna().sum())

koppen_df.to_csv("../../koppen_df_valid.csv")

/var/folders/mf/s1v5mc1n4rdfn5k048ff_5d00000gn/T/ipykernel_37050/2036737967.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  koppen_df["koppen_class"] = koppen


koppen_class
8.0       1109
14.0    182918
15.0     18741
25.0       934
26.0     95329
27.0      2363
Name: count, dtype: int64
Missing Köppen: 0
